In [ ]:
"""
Week 1 - Day 7
Final Cleanup + Summary
========================
Week 1 complete wrap up.
Preparing for Week 2 DQN implementation.

Week 1 Achievements:
✅ Custom Gymnasium Environment
✅ 5 Baseline Agents
✅ Q-Learning Agent
✅ Q-Table Policy Analysis
✅ Unit Tests (8 passing)
✅ Complete Documentation

Infotact DS/ML Internship — Project 2
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

import os
import sys

PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..", "..")
)

SRC_DIR = os.path.join(PROJECT_ROOT, "src")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Current Working Directory:", os.getcwd())
print("Project Root:", PROJECT_ROOT)
print("SRC Directory:", SRC_DIR)
print("SRC exists:", os.path.exists(SRC_DIR))

from environment.pricing_env import (
    DynamicPricingEnv,
    PRICE_LEVELS
)
from agents.baseline_agents import (
    FixedPriceAgent,
    TimedPricingAgent,
    DemandBasedAgent,
    LinearDecayAgent
)
from agents.q_learning_agent import (
    QLearningAgent,
    QL_CONFIG
)
from agents.agent_registry import (
    create_all_baseline_agents,
    create_ql_agent
)
from utils.evaluator import evaluate_agent
from config import ENV, Q_LEARNING, PROJECT_INFO

plt.style.use('seaborn-v0_8')
print("✅ Week 1 Day 7 modules loaded!")
print(f"\nProject: {PROJECT_INFO['name']}")
print(f"Program: {PROJECT_INFO['program']}")

In [ ]:
# Final environment verification
env = DynamicPricingEnv()
obs, _ = env.reset(seed=42)

print("=== ENVIRONMENT FINAL CHECK ===\n")
print(f"✅ Max Inventory  : {env.max_inventory}")
print(f"✅ Max Days       : {env.max_days}")
print(f"✅ Price Levels   : {env.price_levels}")
print(f"✅ Action Space   : {env.action_space}")
print(f"✅ Obs Space      : {env.observation_space}")
print(f"✅ State Space    : "
      f"{(env.max_inventory+1)*(env.max_days+1)}"
      f" states")

# Quick episode test
obs, _ = env.reset(seed=42)
steps  = 0
done   = False
while not done:
    obs, r, term, trunc, info = env.step(
        env.action_space.sample()
    )
    done = term or trunc
    steps += 1

print(f"\n✅ Episode runs correctly ({steps} steps)")
print(f"✅ Environment is Gym-compatible!")

In [ ]:
from tests.test_environment import run_all_tests

print("Running all unit tests...\n")
all_passed = run_all_tests()

if all_passed:
    print("\n🎉 All tests passing!")
else:
    print("\n⚠️ Some tests failed!")

In [ ]:
print("Running final Week 1 comparison...")
print("Training Q-Learning (3000 eps)...\n")

env      = DynamicPricingEnv()
baselines = create_all_baseline_agents(env)
ql_agent  = create_ql_agent(
    env,
    train=True,
    n_episodes=3000
)

# Evaluate all
results = {}
for agent in baselines:
    df = evaluate_agent(agent, n_episodes=100)
    results[agent.name] = df['total_revenue'].mean()

# Q-Learning
ql_eval = ql_agent.evaluate(n_episodes=100)
results['Q-Learning 🤖'] = ql_eval['mean_revenue']

print("\n=== FINAL WEEK 1 RESULTS ===\n")
sorted_results = dict(
    sorted(results.items(),
           key=lambda x: x[1],
           reverse=True)
)
for i, (name, rev) in enumerate(
    sorted_results.items(), 1
):
    medal = "🥇" if i==1 else "🥈" if i==2 else "🥉" if i==3 else "  "
    print(f"  {medal} {name:<20}: ${rev:.0f}")

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig)

# ── Chart 1: Revenue Comparison ──
ax1 = fig.add_subplot(gs[0, :2])
names    = list(sorted_results.keys())
revenues = list(sorted_results.values())
colors   = [
    'gold' if '🤖' in n
    else 'steelblue'
    for n in names
]
bars = ax1.bar(
    names, revenues,
    color=colors,
    edgecolor='black',
    width=0.6
)
ax1.set_title(
    'Week 1 Final — Agent Revenue Comparison',
    fontweight='bold', fontsize=12
)
ax1.set_ylabel('Mean Revenue ($)')
ax1.set_xticklabels(names, rotation=15, fontsize=9)
for bar, val in zip(bars, revenues):
    ax1.text(
        bar.get_x() + bar.get_width()/2,
        val + 20,
        f'${val:.0f}',
        ha='center', fontsize=9,
        fontweight='bold'
    )

# ── Chart 2: Q-Learning Training ──
ax2 = fig.add_subplot(gs[0, 2])
smooth = pd.Series(
    ql_agent.episode_rewards
).rolling(window=100).mean()
ax2.plot(
    ql_agent.episode_rewards,
    alpha=0.2, color='steelblue',
    linewidth=0.5
)
ax2.plot(
    smooth, color='red',
    linewidth=2, label='100-ep avg'
)
ax2.set_title(
    'Q-Learning\nTraining Curve',
    fontweight='bold'
)
ax2.set_xlabel('Episode')
ax2.set_ylabel('Revenue ($)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ── Chart 3: Q-Table Policy ──
ax3 = fig.add_subplot(gs[1, :2])
import seaborn as sns
prices_policy = ql_agent.get_price_policy()
sns.heatmap(
    prices_policy,
    ax=ax3,
    cmap='RdYlGn',
    xticklabels=5,
    yticklabels=5,
    cbar_kws={'label': 'Optimal Price ($)'}
)
ax3.set_title(
    'Learned Q-Table Policy\n'
    'Optimal Price by State',
    fontweight='bold'
)
ax3.set_xlabel('Days Until Departure')
ax3.set_ylabel('Remaining Inventory')

# ── Chart 4: Week 1 Progress ──
ax4 = fig.add_subplot(gs[1, 2])
tasks = [
    'MDP\nDesign',
    'Gym\nEnv',
    'Baselines',
    'Q-Learning',
    'Analysis',
    'Tests'
]
completion = [100, 100, 100, 100, 100, 100]
colors_t   = ['#4CAF50'] * 6
bars_t = ax4.barh(
    tasks, completion,
    color=colors_t,
    edgecolor='black'
)
for bar in bars_t:
    ax4.text(
        50, bar.get_y() + bar.get_height()/2,
        '✅ Done',
        va='center', ha='center',
        fontweight='bold',
        color='white', fontsize=9
    )
ax4.set_title(
    'Week 1 Tasks\nCompletion',
    fontweight='bold'
)
ax4.set_xlabel('Completion %')
ax4.set_xlim(0, 120)

plt.suptitle(
    'Week 1 Final Dashboard\n'
    'RL Dynamic Pricing — Project 2',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    RESULTS_DIR,
    "week1_final_dashboard.png"
)

plt.tight_layout()

plt.savefig(
    SAVE_PATH,
    bbox_inches="tight",
    dpi=150
)

print(f"✅ Saved: {SAVE_PATH}")

plt.show()
plt.show()
print("✅ Week 1 final dashboard saved!")

In [ ]:
print("╔══════════════════════════════════════════╗")
print("║       WEEK 1 — COMPLETE SUMMARY          ║")
print("╠══════════════════════════════════════════╣")
print("║  DELIVERABLES:                           ║")
print("║  ✅ MDP formulated                       ║")
print("║  ✅ Custom Gym Environment               ║")
print("║  ✅ Stochastic demand function           ║")
print("║  ✅ 5 Baseline agents                    ║")
print("║  ✅ Q-Learning agent (5000 eps)          ║")
print("║  ✅ Q-Table analysis + policy            ║")
print("║  ✅ 8 Unit tests passing                 ║")
print("║  ✅ Complete documentation               ║")
print("╠══════════════════════════════════════════╣")
print("║  GITHUB:                                 ║")
print("║  Issues #1 #2 #3 #4 → CLOSED ✅         ║")
print("║  Daily commits maintained ✅             ║")
print("╠══════════════════════════════════════════╣")
print("║  Q-LEARNING LIMITATION:                  ║")
print("║  Works only for small state spaces!      ║")
print("║  Real world = continuous huge states!    ║")
print("║  Solution → Deep Q-Network (DQN)!       ║")
print("╠══════════════════════════════════════════╣")
print("║  WEEK 2 PLAN:                            ║")
print("║  → DQN Architecture (PyTorch)            ║")
print("║  → Experience Replay Buffer              ║")
print("║  → Epsilon Greedy Strategy               ║")
print("║  → Target Network                        ║")
print("║  → DQN vs Q-Learning comparison         ║")
print("╠══════════════════════════════════════════╣")
print("║  🔥 Week 2 starts 12th July!             ║")
print("╚══════════════════════════════════════════╝")